# El barrido de ruido — la corrida que recorre los niveles

Este cuaderno **corre** el barrido y nada más: no arma ninguna tabla ni conclusión — eso vive en `Benchmark_Noise_Report_v1.ipynb`, que lee los `runs.jsonl` que esto deja y dibuja la curva de degradación.

Es una forma distinta de una campaña y no una campaña más chica: **una transferencia recorriendo cada nivel declarado**, contra seis transferencias en un nivel. Por eso escribe bajo `kind="curve"` — las dos pueden pararse en la misma tasa, y `runs.jsonl` se abre en `"w"`.

> **Los techos salen de la búsqueda en limpio y se mantienen fijos en los cinco niveles.** La curva es el coeficiente elegido sin contaminación aplicado con ella, que además es la situación práctica. Lo que eso cuesta es que una caída no se pueda atribuir: el término fallando y el coeficiente quedándose corto se ven igual. Eso lo separa `Benchmark_Noise_Diagnostic_Search_v1.ipynb`, que re-busca a su nivel y no gobierna este registro.
>
> **Se leen del registro, y no se busca nada acá.** Llamar a `harness.with_ceilings_in_force` era el reflejo obvio y es una trampa: cuando no existe `ceilings.json` esa función lanza la búsqueda COMPLETA — dos familias × seis transferencias × treinta trials a veinte épocas, unas nueve horas y media — sin decir que lo está haciendo y sin que nadie la haya autorizado. Un barrido a escala de ensayo no puede ser la puerta por la que entra la corrida larga.
>
> **Guarda pesos en dos de los cinco niveles y en ninguno de los otros tres,** y no lo decide este cuaderno: `campaign()` pregunta por `keeps_checkpoints`, que es verdadero exactamente en los niveles que el cuaderno latente dibuja. Los otros tres corren y escriben sus `runs.jsonl`, que es de donde sale la curva, y ni un peso.

In [ ]:
# Bootstrap: locate the repository wherever this is running, and import from it.
# Local, Colab and Kaggle differ only in where the checkout sits.
import os
import sys
from pathlib import Path


def find_repository() -> Path:
    candidates = [Path.cwd(), *Path.cwd().parents]
    for base in (os.environ.get("MIL_CREDA_REPO", ""), "/content", "/kaggle/working"):
        if base and Path(base).is_dir():
            candidates.append(Path(base))
            candidates.extend(sorted(Path(base).glob("*")))
    for candidate in candidates:
        if (candidate / "src" / "MIL_CREDA_Benchmark").is_dir():
            return candidate.resolve()
    raise SystemExit(
        "cannot find the repository. Set MIL_CREDA_REPO to the checkout that "
        "holds src/MIL_CREDA_Benchmark, or run this notebook from inside it."
    )


REPOSITORY = find_repository()
sys.path.insert(0, str(REPOSITORY / "src"))
print("repository:", REPOSITORY)

In [ ]:
import time
from dataclasses import replace

from MIL_CREDA_Benchmark import config, harness

device = harness.resolve_device()
# La escala configurada decide si esto es un ensayo, y el ensayo decide DÓNDE
# escribe. `config.is_pilot_scale()` es la única lectura de esa regla --- las dos
# constantes que separan una escala de la otra son `EPOCHS` y `SEEDS` ---, y
# escribirla otra vez acá serían dos ortografías de lo mismo. El paso que corre
# este cuaderno se niega cuando esa lectura dice que la escala es la completa,
# porque sus `produces` nombran el árbol de ENSAYO y ninguno más.
ES_ENSAYO = config.is_pilot_scale()

# Sin registro no corre: un barrido con techos vacíos mediría el ruido y la falta
# de coeficiente a la vez. Las dos escalas preguntadas por separado y las dos
# dichas: esta guarda pregunta si existe ALGUNO de los dos archivos, no cuál
# rige. Desde que la omisión significa «el que rige», una llamada pelada
# contestaría por los dos y las dos mitades de la pregunta se volverían una sola.
if (harness.search_record(pilot=False) is None
        and harness.search_record(pilot=True) is None):
    raise SystemExit(
        "no hay registro de techos. Corré primero la búsqueda "
        "(`search-pilot` a escala de ensayo, o la completa con su "
        "autorización): un barrido sin techos mide dos cosas a la vez.")

# Los techos que YA estén en el registro, UNA vez y arriba del bucle: leerlos por
# nivel daría los mismos números hoy y dejaría el barrido a merced de un registro
# que cambie a mitad de corrida.
base = replace(
    harness.Reduction(device=str(device), environment=harness.environment(),
                      pilot=ES_ENSAYO, kind="curve"),
    ceilings=config.ceilings_on_record(),
    ceilingsByTransfer=config.ceilings_by_transfer_on_record())

print(harness.header(base))
print()
print("transferencia:", "{}->{}".format(*config.NOISE_TRANSFER))
print("niveles:", [f"{r:g}" for r in config.NOISE_LEVELS])
print("escala:", "ensayo" if ES_ENSAYO else "completa",
      "| escribe en", config.results_for(0.0, base.kind, base.pilot).parent)

## Lo que cuesta

Una corrida real, cronometrada, antes de comprometerse con los cinco niveles: un
estimado del costo es más barato que el costo, así que va primero. Por los cinco
niveles y no por uno — la rejilla de UNA transferencia corre entera una vez por
nivel, y un pronóstico de un solo nivel diría un quinto de lo que el cuaderno va
a gastar, que es peor que no pronosticar nada.

In [ ]:
from MIL_CREDA_Benchmark import bags

material = {rol: bags.build(codigo, config.DATA_CACHE, config.SEEDS[0])
            for rol, codigo in zip(("source", "target"), config.NOISE_TRANSFER)}
sonda = harness.run_one("G", config.NOISE_TRANSFER, config.SEEDS[0],
                        base, device, material)
por_corrida = sonda["seconds"]
forma = config.sizing()
# La rejilla de UNA transferencia, compuesta desde el aforo y no escrita: brazos
# por semillas, por los niveles declarados.
por_nivel = forma["arms"] * forma["seeds"]
rejilla = por_nivel * len(config.NOISE_LEVELS)
print(f"una corrida completa, {base.epochs} épocas: {por_corrida:.1f}s")
print(f"este barrido ({por_nivel} corridas x {len(config.NOISE_LEVELS)} niveles "
      f"= {rejilla} corridas): unos {rejilla * por_corrida / 60:.0f} min")
del material, sonda

## La corrida

Un nivel por pasada, todas bajo los mismos techos. Lo único que cambia entre una
y la siguiente es la tasa que lleva el material de entrenamiento.

In [ ]:
corridos = {}
for tasa in config.NOISE_LEVELS:
    pasada = replace(base, labelNoise=tasa)
    # De la reducción con la que se está por correr, no de la constante: las
    # coordenadas del primer nivel coinciden con `config.NOISE` por casualidad, y
    # esa casualidad es cómo se lee el árbol equivocado.
    raiz = config.results_for(pasada.labelNoise, pasada.kind, pasada.pilot)
    print(f"\nnivel ρ={tasa:g} → {raiz}")
    empezado = time.perf_counter()
    corridos[f"{tasa:g}"] = harness.campaign(
        pasada, device, transfers=[config.NOISE_TRANSFER])
    print(f"nivel ρ={tasa:g} terminado en "
          f"{(time.perf_counter() - empezado) / 60:.1f} min")

print()
print("la curva y su conclusión las arma Benchmark_Noise_Report_v1.ipynb")